# **Поиск вакансий по резюме. Сравнение**

In [ ]:
# Установка библиотеки (если ещё не установлена)
#!pip install python-docx
#pip install --upgrade transformers
#!pip install google-generativeai

## Проверка моделей суммаризации

In [14]:
with open("file_vac.txt", 'r', encoding='utf-8') as file:
    vacancy_text_test = file.read().strip()

print("Текст вакансии загружен!")
#print(vacancy_text_test)


Текст вакансии загружен!


In [ ]:
from summary_generator import generate_gemini_summary
print(generate_gemini_summary(vacancy_text_test))


{'error': '429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 31.764459187s. [links {\n  description: "Learn more about Gemini API quotas"\n  url: "https://ai.google.dev/gemini-api/docs/rate-limits"\n}\n, violations {\n}\n, retry_delay {\n  seconds: 31\n}\n]'}


In [42]:
from summary_generator import generate_gemini_summary, generate_openai_summary_simple
print(generate_openai_summary_simple(vacancy_text_test))

**Job Summary:**

The role involves the development and maintenance of software applications using the Microsoft technology stack. Key responsibilities include designing, developing, and testing software applications, maintaining systems by identifying and correcting defects, creating technical specifications, and collaborating with various teams (QA, UX, database resources, IT Operations) to deliver user-centric solutions. The position requires adherence to the software development life cycle, coding standards, and methodologies such as Agile/Scrum.

**Minimum Skills and Competencies:**
- Bachelor's Degree or equivalent experience
- 5+ years of experience with Microsoft technology stack
- Expertise in C#, JavaScript, MSSQL 2012 or above
- Proficiency in MVC, Angular, Asp.Net, JQuery
- Experience with Visual Studio & TFS
- Understanding of SOLID OOP concepts and WCF
- End-to-end SDLC project experience
- Strong problem-solving skills and proactive issue resolution
- Effective organizat

In [21]:
PINECONE_API_KEY = "pcsk_2ghPWB_3v4p9rrdZ85hp2SzJnMMEPELbpE1cPiXhXUaiBQueTq7C3Wy1ag271dGJnbbdG7" # акк Анны

vectorizing_model = 'all-MiniLM-L6-v2'
#vectorizing_model = 'sentence-transformers/distilbert-base-nli-mean-tokens'

index_name = 'res-vac-match-summary'
index_dimension=384

# **Часть первая - Наполнение векторной базы**

# Извлечение вакансий из таблицы

In [43]:
import pandas as pd
import os
import time
from summary_generator import generate_gemini_summary, generate_openai_summary_simple

# Путь к папке с датасетом
DATASET_PATH = "vacancy-resume-matching-dataset-main"
csv_file = os.path.join(DATASET_PATH, "5_vacancies.csv")

if not os.path.exists(csv_file):
    raise FileNotFoundError(f"Файл не найден: {csv_file}\nУбедитесь, что папка 'vacancy-resume-matching-dataset-main' находится в текущей директории.")

# Загружаем CSV
df_vacancies = pd.read_csv(csv_file)

# Извлекаем нужные поля и удаляем пустые строки
df_vacancies = df_vacancies[['job_title', 'job_description']].dropna()

# Создаём список chunk_text: каждый элемент — один текст вакансии
chunk_text = []
metadata = []

for idx, row in df_vacancies.iterrows():
    title = row['job_title']
    desc = row['job_description']
    
    # Формируем единый текст для вакансии (один чанк = одна вакансия)
    full_text = f"Job title: {title}. Job description: {desc}"

    # Извлекаем текст саммари для вакансии (один чанк = одна вакансия)
    full_text = generate_openai_summary_simple(full_text)
    print(f'Сгенерировано саммари: {idx+1}')    
    # ЗАДЕРЖКА
    slp = 1
    time.sleep(slp)
    print(f'Прошло секунд задержки: {slp}')
    chunk_text.append(full_text)
    metadata.append({
        "job_title": title,
        "job_description": desc,
        "vacancy_id": idx+1
    })

# Выводим статистику
print(f"Создано {len(chunk_text)} чанков (по одному на вакансию) из {len(df_vacancies)} вакансий.")

# Пример первого чанка
print("\nПример первого чанка:")
print(chunk_text[0])
print("Метаданные:")
print(metadata[0])


Сгенерировано саммари: 1
Прошло секунд задержки: 1
Сгенерировано саммари: 2
Прошло секунд задержки: 1
Сгенерировано саммари: 3
Прошло секунд задержки: 1
Сгенерировано саммари: 4
Прошло секунд задержки: 1
Сгенерировано саммари: 5
Прошло секунд задержки: 1
Создано 5 чанков (по одному на вакансию) из 5 вакансий.

Пример первого чанка:
**Job Title:** Software Developer - .Net

**Key Skills and Requirements:**
- **Education:** Bachelor's Degree or equivalent experience; High school diploma or GED required.
- **Experience:** 5+ years in Microsoft technology stack.
- **Technical Skills:**
  - Expert in C#, JavaScript, MSSQL 2012 or above.
  - Proficient with MVC, Angular, Asp.Net, JQuery.
  - Hands-on experience with Visual Studio & TFS.
  - Understanding of SOLID OOP concepts and experience with WCF.
  - Familiarity with end-to-end Software Development Life Cycle (SDLC) and Agile/Scrum methodologies.
  - Experience with Web API, Entity Framework, Crystal Reports, SSRS, and/or SSIS is desired

## Подготовка к загрузке в Pinecone

In [44]:

from sentence_transformers import SentenceTransformer

# Загружаем модель
model = SentenceTransformer(vectorizing_model)

# Генерируем эмбеддинги
embeddings = model.encode(chunk_text).tolist()  # Список списков, длина 768

# Генерируем уникальные ID для векторов
ids = [f"chunk_{i}" for i in range(len(chunk_text))]

# Подготавливаем записи для upsert
my_chunks = list(zip(ids, embeddings, metadata))

print(f"✅ Готово к загрузке: {len(my_chunks)} векторов в Pinecone")


✅ Готово к загрузке: 5 векторов в Pinecone


## Загрузка 5-и чанков в Pinecone

In [45]:
from pinecone import Pinecone, ServerlessSpec

# Инициализация Pinecone
pc = Pinecone(api_key=PINECONE_API_KEY)


# Создание индекса, если его нет
if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=index_dimension,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )

# Подключение к индексу
index = pc.Index(index_name)

# Загружаем все чанки из my_chunks (всего 5 — можно без батчей)
print(f"📤 Загружаем {len(my_chunks)} векторов в Pinecone...")
upsert_response = index.upsert(vectors=my_chunks)

# Проверка результата
print(f"✅ Успешно загружено {upsert_response['upserted_count']} векторов.")


📤 Загружаем 5 векторов в Pinecone...
✅ Успешно загружено 5 векторов.


# **Часть вторая - Поиск вакансии по резюме**

### Извлечение текста из вордовских файлов

In [46]:
from docx import Document
import os

def extract_text_from_docx(folder_path, file_number):
    """
    Извлекает текст из файла {file_number}.docx в указанной папке.
    Обрабатывает как параграфы, так и текст внутри таблиц.

    Args:
        folder_path (str): Путь к папке с файлами (например, r"C:\CV")
        file_number (int): Номер файла (1 до 65)

    Returns:
        str: Извлечённый текст из .docx файла или пустая строка, если файл не найден/ошибка
    """
    file_name = f"{file_number}.docx"
    file_path = os.path.join(folder_path, file_name)

    try:
        doc = Document(file_path)
        full_text = []

        # 1. Извлекаем текст из параграфов
        for paragraph in doc.paragraphs:
            if paragraph.text.strip():  # Только непустые параграфы
                full_text.append(paragraph.text.strip())

        # 2. Извлекаем текст из таблиц
        for table in doc.tables:
            for row in table.rows:
                for cell in row.cells:
                    for paragraph in cell.paragraphs:
                        if paragraph.text.strip():  # Только непустые параграфы в ячейках
                            full_text.append(paragraph.text.strip())

        # Объединяем все тексты с разрывами строк
        return '\n'.join(full_text)

    except FileNotFoundError:
        print(f"⚠️ Файл не найден: {file_path}")
        return ""
    except Exception as e:
        print(f"❌ Ошибка при чтении {file_path}: {e}")
        return ""


### Генерация поискового списка и поиск

In [47]:
import os
from sentence_transformers import SentenceTransformer
from pinecone import Pinecone

# Инициализация модели для векторизации
model = SentenceTransformer(vectorizing_model)

# Инициализация Pinecone
pc = Pinecone(api_key=PINECONE_API_KEY)
index = pc.Index(index_name)

# Путь к папке CV
CV_FOLDER_PATH = r"vacancy-resume-matching-dataset-main\CV"

# Проверим, существует ли папка
if not os.path.exists(CV_FOLDER_PATH):
    print(f"❌ Папка не существует: {CV_FOLDER_PATH}")
    print("Убедитесь, что путь указан правильно. Возможно, нужно использовать абсолютный путь.")
else:
    print(f"✅ Папка найдена: {CV_FOLDER_PATH}\n")

    # Собираем тексты всех файлов и ищем топ-5 вакансий для каждого
    search_results = []
    
    for i in range(1, 66):  # от 1 до 65 включительно
        print(f"📄 Обработка файла {i}.docx...")
        
        # Извлекаем текст резюме
        text = extract_text_from_docx(CV_FOLDER_PATH, i)
        # ЗАДЕРЖКА
        slp = 1
        time.sleep(slp)
        print(f'Прошло секунд задержки: {slp}')

        # Проверить, что текст файла не пустой
        print(text[:70], "...")
        if text:
            # Векторизуем резюме
            resume_embedding = model.encode([text]).tolist()[0]
            
            # Ищем топ-5 вакансий в Pinecone
            search_response = index.query(
                vector=resume_embedding,
                top_k=5,
                include_metadata=True
            )
            
            # Извлекаем порядковые номера вакансий из metadata (vacancy_id)
            # Вакансии уже отсортированы по релевантности (от самой релевантной к наименее)
            top_5_vacancies = []
            for match in search_response.matches:
                vacancy_id = int(match.metadata["vacancy_id"])  # Преобразуем в целое число
                top_5_vacancies.append(vacancy_id)
            
            search_results.append(top_5_vacancies)
            print(f"   ✅ Найдено вакансий: {top_5_vacancies}")
        else:
            # Если текст не извлекся, добавляем пустой список
            search_results.append([])
            print(f"   ⚠️ Текст не извлечен")

    print(f"\n=== Готово! ===")
    print(f"Обработано резюме: {len([r for r in search_results if r])} из 65")
    


✅ Папка найдена: vacancy-resume-matching-dataset-main\CV

📄 Обработка файла 1.docx...
Прошло секунд задержки: 1
****************
Java full stack developer
(60% – IN JAVA BACKEND DEVE ...
   ✅ Найдено вакансий: [1, 3, 2, 5, 4]
📄 Обработка файла 2.docx...
Прошло секунд задержки: 1
*************
FULL-STACK DEVELOPER
4 years’ experience (80 % in JAVA B ...
   ✅ Найдено вакансий: [1, 3, 2, 5, 4]
📄 Обработка файла 3.docx...
Прошло секунд задержки: 1
*************************
5 years of total experience in development w ...
   ✅ Найдено вакансий: [1, 3, 2, 5, 4]
📄 Обработка файла 4.docx...
Прошло секунд задержки: 1
**************
JAVA BACKEND DEVELOPER WITH 3 YEARS OF HANDS-ON EXPERIE ...
   ✅ Найдено вакансий: [1, 3, 2, 5, 4]
📄 Обработка файла 5.docx...
Прошло секунд задержки: 1
6 years of proven experience
(4 years in Java Backend/1 year in NodeJS ...
   ✅ Найдено вакансий: [1, 3, 2, 5, 4]
📄 Обработка файла 6.docx...
Прошло секунд задержки: 1
LEONID SHALAEV FULL STACK DEVELOPER with 1+ year

In [39]:
from dotenv import load_dotenv
load_dotenv()
# summarizing_model = os.getenv("gemini_model")
summarizing_model = os.getenv("openai_model")

In [48]:
# Выводим структуру результатов (аналогично ANNOTATOR_1_RANKINGS)
print(f"\n📊 Результаты поиска (модель для саммари - {summarizing_model}, модель векторизации - {vectorizing_model}, размер - {index_dimension}):")
print("SEARCH_RESULTS = [\n", end="")
for i in range(0, len(search_results), 5):
    group = search_results[i:i+5]
    
    # Формируем комментарий
    start_num = i + 1
    end_num = min(i + 5, len(search_results))
    comment = f"#{start_num}-{end_num}" if end_num - start_num + 1 == 5 else f"#{start_num}-{end_num}"
    
    # Выводим первую строку группы с комментарием
    if i == 0:
        print(f"{group[0]},{group[1]},{group[2]},{group[3]},{group[4]}, {comment}")
    else:
        print(f"{group[0]},{group[1]},{group[2]},{group[3]},{group[4]}, {comment}")
print("]")



📊 Результаты поиска (модель для саммари - openai/gpt-4o-mini, модель векторизации - all-MiniLM-L6-v2, размер - 384):
SEARCH_RESULTS = [
[1, 3, 2, 5, 4],[1, 3, 2, 5, 4],[1, 3, 2, 5, 4],[1, 3, 2, 5, 4],[1, 3, 2, 5, 4], #1-5
[1, 3, 2, 5, 4],[1, 3, 2, 5, 4],[1, 3, 2, 5, 4],[1, 3, 2, 5, 4],[1, 3, 2, 5, 4], #6-10
[1, 3, 2, 5, 4],[1, 3, 2, 5, 4],[1, 3, 2, 5, 4],[1, 3, 2, 5, 4],[1, 3, 2, 5, 4], #11-15
[1, 3, 2, 5, 4],[1, 3, 2, 5, 4],[1, 3, 2, 5, 4],[1, 3, 2, 5, 4],[1, 3, 2, 5, 4], #16-20
[1, 3, 2, 5, 4],[1, 3, 2, 5, 4],[1, 3, 2, 5, 4],[1, 3, 2, 5, 4],[1, 3, 2, 5, 4], #21-25
[1, 3, 2, 5, 4],[1, 3, 2, 5, 4],[1, 3, 2, 5, 4],[1, 3, 2, 5, 4],[1, 3, 2, 5, 4], #26-30
[1, 3, 2, 5, 4],[1, 3, 2, 5, 4],[1, 3, 2, 5, 4],[1, 3, 2, 5, 4],[1, 3, 2, 5, 4], #31-35
[1, 3, 2, 5, 4],[3, 4, 5, 1, 2],[1, 3, 4, 5, 2],[4, 3, 2, 5, 1],[4, 5, 3, 1, 2], #36-40
[4, 3, 1, 5, 2],[3, 4, 2, 5, 1],[4, 3, 2, 5, 1],[4, 3, 2, 5, 1],[3, 4, 2, 1, 5], #41-45
[4, 5, 3, 1, 2],[1, 4, 3, 5, 2],[4, 3, 1, 5, 2],[4, 3, 5, 2, 1],[1, 4, 2, 

In [31]:
# Выводим структуру результатов (аналогично ANNOTATOR_1_RANKINGS)
print(f"\n📊 Результаты поиска (модель для саммари - {summarizing_model}, модель векторизации - {vectorizing_model}, размер - {index_dimension}):")
print("SEARCH_RESULTS = [\n", end="")
for i in range(0, len(search_results), 5):
    group = search_results[i:i+5]
    
    # Формируем комментарий
    start_num = i + 1
    end_num = min(i + 5, len(search_results))
    comment = f"#{start_num}-{end_num}" if end_num - start_num + 1 == 5 else f"#{start_num}-{end_num}"
    
    # Выводим первую строку группы с комментарием
    if i == 0:
        print(f"{group[0]},{group[1]},{group[2]},{group[3]},{group[4]}, {comment}")
    else:
        print(f"{group[0]},{group[1]},{group[2]},{group[3]},{group[4]}, {comment}")
print("]")



📊 Результаты поиска (модель для саммари - openai/gpt-4o-mini, модель векторизации - all-MiniLM-L6-v2, размер - 384):
SEARCH_RESULTS = [
[1, 2, 4, 5, 3],[1, 2, 4, 5, 3],[1, 2, 4, 5, 3],[1, 2, 4, 5, 3],[1, 2, 4, 5, 3], #1-5
[1, 2, 4, 5, 3],[1, 2, 4, 5, 3],[1, 2, 4, 5, 3],[1, 2, 4, 5, 3],[1, 2, 4, 5, 3], #6-10
[1, 2, 4, 5, 3],[1, 2, 4, 5, 3],[1, 2, 4, 5, 3],[1, 2, 4, 5, 3],[1, 2, 4, 5, 3], #11-15
[1, 2, 4, 5, 3],[1, 2, 4, 5, 3],[1, 2, 4, 5, 3],[1, 2, 4, 5, 3],[1, 2, 4, 5, 3], #16-20
[1, 2, 4, 5, 3],[1, 2, 4, 5, 3],[1, 2, 4, 5, 3],[1, 2, 4, 5, 3],[1, 4, 5, 3, 2], #21-25
[4, 3, 5, 2, 1],[4, 5, 3, 2, 1],[4, 5, 3, 1, 2],[4, 5, 3, 2, 1],[5, 4, 3, 2, 1], #26-30
[4, 5, 3, 1, 2],[3, 4, 1, 5, 2],[4, 3, 2, 5, 1],[4, 3, 5, 2, 1],[4, 5, 1, 3, 2], #31-35
[3, 5, 4, 1, 2],[5, 3, 4, 2, 1],[3, 1, 4, 5, 2],[4, 3, 2, 5, 1],[4, 5, 3, 1, 2], #36-40
[4, 3, 1, 5, 2],[4, 3, 2, 5, 1],[4, 3, 5, 2, 1],[4, 3, 2, 5, 1],[3, 4, 2, 5, 1], #41-45
[4, 5, 3, 1, 2],[1, 3, 4, 5, 2],[4, 3, 1, 5, 2],[4, 5, 3, 2, 1],[1, 3, 4, 

In [40]:
# Выводим структуру результатов (аналогично ANNOTATOR_1_RANKINGS)
print(f"\n📊 Результаты поиска (модель для саммари - {summarizing_model}, модель векторизации - {vectorizing_model}, размер - {index_dimension}):")
print("SEARCH_RESULTS = [\n", end="")
for i in range(0, len(search_results), 5):
    group = search_results[i:i+5]
    
    # Формируем комментарий
    start_num = i + 1
    end_num = min(i + 5, len(search_results))
    comment = f"#{start_num}-{end_num}" if end_num - start_num + 1 == 5 else f"#{start_num}-{end_num}"
    
    # Выводим первую строку группы с комментарием
    if i == 0:
        print(f"{group[0]},{group[1]},{group[2]},{group[3]},{group[4]}, {comment}")
    else:
        print(f"{group[0]},{group[1]},{group[2]},{group[3]},{group[4]}, {comment}")
print("]")



📊 Результаты поиска (модель для саммари - openai/gpt-4o-mini, модель векторизации - all-MiniLM-L6-v2, размер - 384):
SEARCH_RESULTS = [
[5, 4, 1, 2, 3],[5, 4, 1, 2, 3],[5, 4, 1, 2, 3],[5, 4, 1, 2, 3],[5, 4, 1, 2, 3], #1-5
[5, 4, 1, 2, 3],[5, 4, 1, 2, 3],[5, 4, 1, 2, 3],[5, 4, 1, 2, 3],[5, 4, 1, 2, 3], #6-10
[5, 4, 1, 2, 3],[5, 4, 1, 2, 3],[5, 4, 1, 2, 3],[5, 4, 1, 2, 3],[5, 4, 1, 2, 3], #11-15
[5, 4, 1, 2, 3],[5, 4, 1, 2, 3],[5, 4, 1, 2, 3],[5, 4, 1, 2, 3],[5, 4, 1, 2, 3], #16-20
[5, 4, 1, 2, 3],[5, 4, 1, 2, 3],[5, 4, 1, 2, 3],[5, 4, 1, 2, 3],[5, 4, 1, 2, 3], #21-25
[5, 4, 1, 2, 3],[5, 4, 1, 2, 3],[5, 4, 1, 2, 3],[5, 4, 1, 2, 3],[5, 4, 1, 2, 3], #26-30
[5, 4, 1, 2, 3],[5, 4, 1, 2, 3],[5, 4, 1, 2, 3],[5, 4, 1, 2, 3],[5, 4, 1, 2, 3], #31-35
[3, 4, 5, 1, 2],[4, 3, 5, 1, 2],[3, 1, 5, 4, 2],[4, 3, 5, 2, 1],[4, 5, 3, 1, 2], #36-40
[4, 3, 1, 5, 2],[4, 3, 2, 5, 1],[4, 3, 5, 2, 1],[4, 3, 5, 2, 1],[3, 4, 2, 1, 5], #41-45
[4, 5, 3, 1, 2],[1, 3, 4, 5, 2],[3, 4, 1, 5, 2],[4, 5, 3, 1, 2],[1, 3, 4, 

### 🔥 Удаление индекса 

In [49]:
# Удалить индекс через клиент
pc.delete_index(index_name)

print(f"✅ Индекс '{index_name}' успешно удалён.")


✅ Индекс 'res-vac-match-summary' успешно удалён.


In [ ]:


📊 Результаты поиска 3 (модель для саммари - openai/gpt-4o-mini, модель векторизации - all-MiniLM-L6-v2, размер - 384):
SEARCH_RESULTS = [
[1, 3, 2, 5, 4],[1, 3, 2, 5, 4],[1, 3, 2, 5, 4],[1, 3, 2, 5, 4],[1, 3, 2, 5, 4], #1-5
[1, 3, 2, 5, 4],[1, 3, 2, 5, 4],[1, 3, 2, 5, 4],[1, 3, 2, 5, 4],[1, 3, 2, 5, 4], #6-10
[1, 3, 2, 5, 4],[1, 3, 2, 5, 4],[1, 3, 2, 5, 4],[1, 3, 2, 5, 4],[1, 3, 2, 5, 4], #11-15
[1, 3, 2, 5, 4],[1, 3, 2, 5, 4],[1, 3, 2, 5, 4],[1, 3, 2, 5, 4],[1, 3, 2, 5, 4], #16-20
[1, 3, 2, 5, 4],[1, 3, 2, 5, 4],[1, 3, 2, 5, 4],[1, 3, 2, 5, 4],[1, 3, 2, 5, 4], #21-25
[1, 3, 2, 5, 4],[1, 3, 2, 5, 4],[1, 3, 2, 5, 4],[1, 3, 2, 5, 4],[1, 3, 2, 5, 4], #26-30
[1, 3, 2, 5, 4],[1, 3, 2, 5, 4],[1, 3, 2, 5, 4],[1, 3, 2, 5, 4],[1, 3, 2, 5, 4], #31-35
[1, 3, 2, 5, 4],[3, 4, 5, 1, 2],[1, 3, 4, 5, 2],[4, 3, 2, 5, 1],[4, 5, 3, 1, 2], #36-40
[4, 3, 1, 5, 2],[3, 4, 2, 5, 1],[4, 3, 2, 5, 1],[4, 3, 2, 5, 1],[3, 4, 2, 1, 5], #41-45
[4, 5, 3, 1, 2],[1, 4, 3, 5, 2],[4, 3, 1, 5, 2],[4, 3, 5, 2, 1],[1, 4, 2, 3, 5], #46-50
[5, 3, 1, 2, 4],[4, 1, 5, 3, 2],[5, 3, 1, 2, 4],[3, 5, 4, 2, 1],[4, 3, 1, 2, 5], #51-55
[1, 4, 3, 2, 5],[4, 2, 3, 5, 1],[4, 1, 5, 3, 2],[3, 2, 4, 5, 1],[4, 3, 1, 5, 2], #56-60
[4, 3, 1, 2, 5],[3, 4, 2, 5, 1],[3, 4, 1, 2, 5],[4, 3, 1, 5, 2],[4, 2, 1, 3, 5], #61-65
]


📊 Результаты поиска 2 (модель для саммари - openai/gpt-4o-mini, модель векторизации - all-MiniLM-L6-v2, размер - 384):
SEARCH_RESULTS = [
[5, 4, 1, 2, 3],[5, 4, 1, 2, 3],[5, 4, 1, 2, 3],[5, 4, 1, 2, 3],[5, 4, 1, 2, 3], #1-5
[5, 4, 1, 2, 3],[5, 4, 1, 2, 3],[5, 4, 1, 2, 3],[5, 4, 1, 2, 3],[5, 4, 1, 2, 3], #6-10
[5, 4, 1, 2, 3],[5, 4, 1, 2, 3],[5, 4, 1, 2, 3],[5, 4, 1, 2, 3],[5, 4, 1, 2, 3], #11-15
[5, 4, 1, 2, 3],[5, 4, 1, 2, 3],[5, 4, 1, 2, 3],[5, 4, 1, 2, 3],[5, 4, 1, 2, 3], #16-20
[5, 4, 1, 2, 3],[5, 4, 1, 2, 3],[5, 4, 1, 2, 3],[5, 4, 1, 2, 3],[5, 4, 1, 2, 3], #21-25
[5, 4, 1, 2, 3],[5, 4, 1, 2, 3],[5, 4, 1, 2, 3],[5, 4, 1, 2, 3],[5, 4, 1, 2, 3], #26-30
[5, 4, 1, 2, 3],[5, 4, 1, 2, 3],[5, 4, 1, 2, 3],[5, 4, 1, 2, 3],[5, 4, 1, 2, 3], #31-35
[3, 4, 5, 1, 2],[4, 3, 5, 1, 2],[3, 1, 5, 4, 2],[4, 3, 5, 2, 1],[4, 5, 3, 1, 2], #36-40
[4, 3, 1, 5, 2],[4, 3, 2, 5, 1],[4, 3, 5, 2, 1],[4, 3, 5, 2, 1],[3, 4, 2, 1, 5], #41-45
[4, 5, 3, 1, 2],[1, 3, 4, 5, 2],[3, 4, 1, 5, 2],[4, 5, 3, 1, 2],[1, 3, 4, 2, 5], #46-50
[5, 3, 1, 2, 4],[4, 1, 5, 3, 2],[5, 3, 1, 2, 4],[3, 4, 5, 1, 2],[4, 3, 1, 5, 2], #51-55
[1, 4, 3, 5, 2],[4, 2, 3, 5, 1],[1, 4, 5, 3, 2],[3, 2, 4, 5, 1],[4, 1, 3, 5, 2], #56-60
[4, 3, 1, 5, 2],[3, 4, 2, 5, 1],[3, 4, 1, 2, 5],[4, 1, 3, 5, 2],[2, 3, 5, 4, 1], #61-65
]


📊 Результаты поиска 1 (модель для саммари - openai/gpt-4o-mini, модель векторизации - all-MiniLM-L6-v2, размер - 384):
SEARCH_RESULTS = [
[1, 2, 4, 5, 3],[1, 2, 4, 5, 3],[1, 2, 4, 5, 3],[1, 2, 4, 5, 3],[1, 2, 4, 5, 3], #1-5
[1, 2, 4, 5, 3],[1, 2, 4, 5, 3],[1, 2, 4, 5, 3],[1, 2, 4, 5, 3],[1, 2, 4, 5, 3], #6-10
[1, 2, 4, 5, 3],[1, 2, 4, 5, 3],[1, 2, 4, 5, 3],[1, 2, 4, 5, 3],[1, 2, 4, 5, 3], #11-15
[1, 2, 4, 5, 3],[1, 2, 4, 5, 3],[1, 2, 4, 5, 3],[1, 2, 4, 5, 3],[1, 2, 4, 5, 3], #16-20
[1, 2, 4, 5, 3],[1, 2, 4, 5, 3],[1, 2, 4, 5, 3],[1, 2, 4, 5, 3],[1, 4, 5, 3, 2], #21-25
[4, 3, 5, 2, 1],[4, 5, 3, 2, 1],[4, 5, 3, 1, 2],[4, 5, 3, 2, 1],[5, 4, 3, 2, 1], #26-30
[4, 5, 3, 1, 2],[3, 4, 1, 5, 2],[4, 3, 2, 5, 1],[4, 3, 5, 2, 1],[4, 5, 1, 3, 2], #31-35
[3, 5, 4, 1, 2],[5, 3, 4, 2, 1],[3, 1, 4, 5, 2],[4, 3, 2, 5, 1],[4, 5, 3, 1, 2], #36-40
[4, 3, 1, 5, 2],[4, 3, 2, 5, 1],[4, 3, 5, 2, 1],[4, 3, 2, 5, 1],[3, 4, 2, 5, 1], #41-45
[4, 5, 3, 1, 2],[1, 3, 4, 5, 2],[4, 3, 1, 5, 2],[4, 5, 3, 2, 1],[1, 3, 4, 5, 2], #46-50
[5, 3, 2, 1, 4],[4, 1, 5, 3, 2],[5, 3, 2, 1, 4],[3, 5, 4, 2, 1],[4, 3, 5, 1, 2], #51-55
[1, 4, 5, 3, 2],[4, 2, 5, 3, 1],[1, 4, 5, 2, 3],[3, 2, 5, 4, 1],[4, 3, 5, 1, 2], #56-60
[4, 3, 1, 5, 2],[3, 2, 4, 5, 1],[3, 4, 1, 5, 2],[4, 1, 5, 3, 2],[2, 4, 5, 3, 1], #61-65
]








📊 Результаты поиска с внешней токенизацией по 3-граммам (модель - all-MiniLM-L6-v2, размер - 384):
SEARCH_RESULTS = [
[1, 2, 3, 5, 4],[2, 5, 4, 1, 3],[2, 5, 1, 3, 4],[5, 2, 3, 4, 1],[2, 5, 4, 3, 1], #1-5
[4, 5, 2, 1, 3],[5, 2, 4, 1, 3],[2, 5, 4, 3, 1],[2, 4, 5, 1, 3],[5, 3, 2, 1, 4], #6-10
[5, 3, 4, 2, 1],[4, 1, 2, 5, 3],[2, 4, 5, 3, 1],[2, 5, 3, 4, 1],[2, 4, 5, 1, 3], #11-15
[2, 5, 3, 1, 4],[5, 2, 3, 4, 1],[2, 3, 5, 1, 4],[2, 5, 4, 3, 1],[5, 2, 4, 3, 1], #16-20
[5, 2, 3, 4, 1],[2, 5, 3, 4, 1],[2, 4, 3, 5, 1],[4, 1, 5, 3, 2],[4, 2, 5, 3, 1], #21-25
[3, 2, 5, 1, 4],[2, 5, 4, 1, 3],[4, 2, 3, 1, 5],[2, 5, 4, 3, 1],[5, 2, 4, 3, 1], #26-30
[2, 5, 4, 1, 3],[5, 2, 4, 1, 3],[5, 2, 4, 1, 3],[5, 2, 1, 4, 3],[2, 4, 3, 5, 1], #31-35
[5, 4, 1, 2, 3],[5, 2, 4, 1, 3],[5, 2, 1, 3, 4],[5, 2, 1, 4, 3],[2, 5, 4, 3, 1], #36-40
[2, 5, 4, 1, 3],[3, 2, 1, 5, 4],[2, 4, 5, 3, 1],[2, 5, 4, 3, 1],[5, 2, 4, 1, 3], #41-45
[4, 2, 5, 3, 1],[5, 2, 4, 1, 3],[2, 5, 3, 4, 1],[5, 2, 1, 3, 4],[3, 1, 2, 5, 4], #46-50
[3, 2, 5, 1, 4],[5, 2, 4, 1, 3],[3, 2, 1, 5, 4],[5, 2, 1, 3, 4],[2, 5, 3, 1, 4], #51-55
[5, 2, 3, 4, 1],[5, 2, 4, 3, 1],[2, 3, 1, 5, 4],[2, 5, 4, 1, 3],[5, 3, 1, 2, 4], #56-60
[3, 1, 5, 4, 2],[3, 2, 5, 1, 4],[2, 5, 1, 3, 4],[2, 3, 5, 4, 1],[5, 4, 2, 3, 1], #61-65
]

📊 Результаты поиска с внешней токенизацией по 3-граммам (модель - all-MiniLM-L6-v2, размер - 384):
SEARCH_RESULTS = [
[1, 2, 3, 5, 4],[2, 5, 4, 1, 3],[2, 5, 1, 3, 4],[5, 2, 3, 4, 1],[2, 5, 4, 3, 1], #1-5
[4, 5, 2, 1, 3],[5, 2, 4, 1, 3],[2, 5, 4, 3, 1],[2, 4, 5, 1, 3],[5, 3, 2, 1, 4], #6-10
[5, 3, 4, 2, 1],[4, 1, 2, 5, 3],[2, 4, 5, 3, 1],[2, 5, 3, 4, 1],[2, 4, 5, 1, 3], #11-15
[2, 5, 3, 1, 4],[5, 2, 3, 4, 1],[2, 3, 5, 1, 4],[2, 5, 4, 3, 1],[5, 2, 4, 3, 1], #16-20
[5, 2, 3, 4, 1],[2, 5, 3, 4, 1],[2, 4, 3, 5, 1],[4, 1, 5, 3, 2],[4, 2, 5, 3, 1], #21-25
[3, 2, 5, 1, 4],[2, 5, 4, 1, 3],[4, 2, 3, 1, 5],[2, 5, 4, 3, 1],[5, 2, 4, 3, 1], #26-30
[2, 5, 4, 1, 3],[5, 2, 4, 1, 3],[5, 2, 4, 1, 3],[5, 2, 1, 4, 3],[2, 4, 3, 5, 1], #31-35
[5, 4, 1, 2, 3],[5, 2, 4, 1, 3],[5, 2, 1, 3, 4],[5, 2, 1, 4, 3],[2, 5, 4, 3, 1], #36-40
[2, 5, 4, 1, 3],[3, 2, 1, 5, 4],[2, 4, 5, 3, 1],[2, 5, 4, 3, 1],[5, 2, 4, 1, 3], #41-45
[4, 2, 5, 3, 1],[5, 2, 4, 1, 3],[2, 5, 3, 4, 1],[5, 2, 1, 3, 4],[3, 1, 2, 5, 4], #46-50
[3, 2, 5, 1, 4],[5, 2, 4, 1, 3],[3, 2, 1, 5, 4],[5, 2, 1, 3, 4],[2, 5, 3, 1, 4], #51-55
[5, 2, 3, 4, 1],[5, 2, 4, 3, 1],[2, 3, 1, 5, 4],[2, 5, 4, 1, 3],[5, 3, 1, 2, 4], #56-60
[3, 1, 5, 4, 2],[3, 2, 5, 1, 4],[2, 5, 1, 3, 4],[2, 3, 5, 4, 1],[5, 4, 2, 3, 1], #61-65
]